# Imports and setup

In [31]:
import requests, warnings
import pandas as pd
import io
from google.colab import files
from time import sleep
from pathlib import Path
from datetime import datetime

In [4]:
met_metadata_df = pd.read_csv(io.StringIO('''
ID,Start date,End Date,Global ID
0,1994,2001,72520514762
1,2009,2012,72219013874
2,2005,2007,07481099999
3,2013,2015,72534014819
4,2016,2017,76679399999
5,2019,2022,76679399999
6,2021,2025,72572024127
7,1992,2005,72494523293
8,2010,2019,47113199999
9,2010,2016,47113199999
10,2006,2014,74486094789
11,1995,2006,07157099999
12,2018,2022,72422093820
13,2007,2012,72640014839
14,1993,2000,72658014922
15,2003,2007,83775399999
16,1988,2000,07157099999
17,1991,2008,72509014739
18,2015,2017,47113199999
19,2004,2020,06180099999
20,2010,2012,06180099999
21,2012,2015,94767099999
'''), header=None)

In [6]:
met_metadata_df.columns = met_metadata_df.iloc[0]
met_metadata_df = met_metadata_df[1:]

In [7]:
met_metadata_df

,ID,Start date,End Date,Global ID
1,0,1994,2001,72520514762
2,1,2009,2012,72219013874
3,2,2005,2007,07481099999
4,3,2013,2015,72534014819
5,4,2016,2017,76679399999
6,5,2019,2022,76679399999
7,6,2021,2025,72572024127
8,7,1992,2005,72494523293
9,8,2010,2019,47113199999
10,9,2010,2016,47113199999


In [8]:
met_metadata_df.columns

Index(['ID', 'Start date', 'End Date', 'Global ID'], dtype='object', name=0)

# Validity Test 1: if station exists

In [9]:
BASE_URL = "https://www.ncei.noaa.gov/access/services/search/v1/autocomplete"

results = []

In [10]:
for station_id in met_metadata_df["Global ID"].astype(str):
    params = {
        "field": "stations",
        "dataset": "global-hourly",
        "text": station_id
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=10)
        r.raise_for_status()

        data = r.json()

        is_valid = len(data.get("results", [])) > 0

        # Grab station name if valid
        station_name = (
            data["results"][0]["name"]
            if is_valid else None
        )

        results.append({
            "station_id": station_id,
            "valid": is_valid,
            "station_name": station_name
        })

    except Exception as e:
        results.append({
            "station_id": station_id,
            "valid": False,
            "station_name": None,
            "error": str(e)
        })

    # polite pause to avoid hammering API
    sleep(0.1)

In [11]:
# Convert to DataFrame
results_df = pd.DataFrame(results)

In [12]:
results_df

,station_id,valid,station_name
0,72520514762,True,"PITTSBURGH ALLEGHENY CO AIRPORT, PA US"
1,72219013874,True,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...
2,07481099999,True,"LYON SAINT EXUPERY, FR"
3,72534014819,True,"CHICAGO MIDWAY AIRPORT, IL US"
4,76679399999,True,LICENCIADO BENITO JUAREZ INTERNATIONAL MEXICO ...
5,76679399999,True,LICENCIADO BENITO JUAREZ INTERNATIONAL MEXICO ...
6,72572024127,True,"SALT LAKE CITY INTERNATIONAL AIRPORT, UT US"
7,72494523293,True,"SAN JOSE, CA US"
8,47113199999,True,"INCHEON INTERNATIONAL, KS"
9,47113199999,True,"INCHEON INTERNATIONAL, KS"


# Validity Test 2: fetch the station metadata, also retrieve coordinates

In [13]:
def fetch_top_left(station_id):
    url = (
        "https://www.ncei.noaa.gov/access/services/search/v1/data"
        f"?dataset=global-hourly&stations={station_id}"
    )

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        top_left = data.get("bounds", {}).get("topLeft", {})

        return pd.Series({
            "top_left_lat": top_left.get("lat"),
            "top_left_lon": top_left.get("lon")
        })

    except Exception:
        return pd.Series({
            "top_left_lat": None,
            "top_left_lon": None
        })

In [14]:
# Add coordinates to dataframe
coords = met_metadata_df["Global ID"].apply(fetch_top_left)

results_df_2 = pd.concat([met_metadata_df, coords], axis=1)

In [15]:
results_df_2

,ID,Start date,End Date,Global ID,top_left_lat,top_left_lon
1,0,1994,2001,72520514762,40.355100,-79.921670
2,1,2009,2012,72219013874,33.630100,-84.442240
3,2,2005,2007,07481099999,45.726387,5.090833
4,3,2013,2015,72534014819,41.786110,-87.755140
5,4,2016,2017,76679399999,19.436303,-99.072097
6,5,2019,2022,76679399999,19.436303,-99.072097
7,6,2021,2025,72572024127,40.778100,-111.969400
8,7,1992,2005,72494523293,37.359380,-121.924440
9,8,2010,2019,47113199999,37.469075,126.450517
10,9,2010,2016,47113199999,37.469075,126.450517


In [19]:
print(results_df_2[['top_left_lat', 'top_left_lon']])

    top_left_lat  top_left_lon
1      40.355100    -79.921670
2      33.630100    -84.442240
3      45.726387      5.090833
4      41.786110    -87.755140
5      19.436303    -99.072097
6      19.436303    -99.072097
7      40.778100   -111.969400
8      37.359380   -121.924440
9      37.469075    126.450517
10     37.469075    126.450517
11     40.639150    -73.764010
12     49.012779      2.550000
13     38.040800    -84.611380
14     42.955000    -87.904570
15     44.885230    -93.231330
16    -23.433000    -46.467000
17     49.012779      2.550000
18     42.360600    -71.009750
19     37.469075    126.450517
20     55.617917     12.655972
21     55.617917     12.655972
22    -33.946111    151.177222


# Actually pull the met data now

In [39]:
BASE_URL = "https://www.ncei.noaa.gov/access/services/data/v1"

In [40]:
met_metadata_df

,ID,Start date,End Date,Global ID
1,0,1994,2001,72520514762
2,1,2009,2012,72219013874
3,2,2005,2007,07481099999
4,3,2013,2015,72534014819
5,4,2016,2017,76679399999
6,5,2019,2022,76679399999
7,6,2021,2025,72572024127
8,7,1992,2005,72494523293
9,8,2010,2019,47113199999
10,9,2010,2016,47113199999


In [41]:
met_metadata_df.columns

Index(['ID', 'Start date', 'End Date', 'Global ID'], dtype='object', name=0)

In [42]:
# Create output directory
output_dir = Path("met_csvs")
output_dir.mkdir(exist_ok=True)

In [43]:
download_results = []

In [44]:
current_year = datetime.now().year

In [45]:
for _, row in met_metadata_df.iterrows():

    # -----------------------------
    # Extract row values
    # -----------------------------
    id_value = str(row["ID"]).strip()

    # Skip ID 15 entirely
    if id_value == "15":
        print("Skipping ID 15")
        continue

    station = str(row["Global ID"]).strip()

    # Convert years from string -> int
    start_year = int(float(row["Start date"]))
    end_year = int(float(row["End Date"]))

    # Expand by +/- 5 years
    request_start = start_year - 5
    request_end = min(end_year + 5, current_year)

    print(f"\nFetching station {station} for ID {id_value}")
    print(f"Date range: {request_start} -> {request_end}")

    params = {
        "dataset": "global-hourly",
        "format": "csv",
        "units": "metric",
        "stations": station,
        "startDate": f"{request_start}-01-01",
        "endDate": f"{request_end}-12-31",
        "dataTypes": (
            "DATE,TMP,DEW,SLP,WND,AA1,GA1,GF1"
        ),
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=120)

        # Raise HTTP errors
        r.raise_for_status()

        # NOAA sometimes returns empty response
        if not r.text.strip():
            warnings.warn(f"No data returned for station: {station}")

            download_results.append({
                "ID": id_value,
                "station": station,
                "success": False,
                "rows": 0,
                "error": "Empty response"
            })

            continue

        # Read CSV into DataFrame
        df = pd.read_csv(
            io.StringIO(r.text),
            dtype=str,
            low_memory=False
        )

        # Handle accidental empty CSV
        if df.empty:
            warnings.warn(f"Empty dataframe for station: {station}")

            download_results.append({
                "ID": id_value,
                "station": station,
                "success": False,
                "rows": 0,
                "error": "Empty dataframe"
            })

            continue

        # -----------------------------------------
        # Save CSV
        # -----------------------------------------
        output_path = output_dir / f"{id_value}_met.csv"

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")
        print(f"Rows: {len(df)}")

        download_results.append({
            "ID": id_value,
            "station": station,
            "success": True,
            "rows": len(df),
            "error": None
        })

    except Exception as e:

        print(f"FAILED for ID {id_value}: {e}")

        download_results.append({
            "ID": id_value,
            "station": station,
            "success": False,
            "rows": 0,
            "error": str(e)
        })

    # Be polite to NOAA
    sleep(0.2)


Fetching station 72520514762 for ID 0
Date range: 1989 -> 2006
Saved: met_csvs/0_met.csv
Rows: 179032

Fetching station 72219013874 for ID 1
Date range: 2004 -> 2017
Saved: met_csvs/1_met.csv
Rows: 183703

Fetching station 07481099999 for ID 2
Date range: 2000 -> 2012
Saved: met_csvs/2_met.csv
Rows: 206905

Fetching station 72534014819 for ID 3
Date range: 2008 -> 2020
Saved: met_csvs/3_met.csv
Rows: 162257

Fetching station 76679399999 for ID 4
Date range: 2011 -> 2022
Saved: met_csvs/4_met.csv
Rows: 143684

Fetching station 76679399999 for ID 5
Date range: 2014 -> 2026
Saved: met_csvs/5_met.csv
Rows: 134975

Fetching station 72572024127 for ID 6
Date range: 2016 -> 2026
Saved: met_csvs/6_met.csv
Rows: 127136

Fetching station 72494523293 for ID 7
Date range: 1987 -> 2010
Saved: met_csvs/7_met.csv
Rows: 285710

Fetching station 47113199999 for ID 8
Date range: 2005 -> 2024
Saved: met_csvs/8_met.csv
Rows: 349351

Fetching station 47113199999 for ID 9
Date range: 2005 -> 2021
Saved: me

In [46]:
# -----------------------------------------
# Save download summary
# -----------------------------------------
download_df = pd.DataFrame(download_results)

In [47]:
download_df

,ID,station,success,rows,error
0,0,72520514762,True,179032,None
1,1,72219013874,True,183703,None
2,2,07481099999,True,206905,None
3,3,72534014819,True,162257,None
4,4,76679399999,True,143684,None
5,5,76679399999,True,134975,None
6,6,72572024127,True,127136,None
7,7,72494523293,True,285710,None
8,8,47113199999,True,349351,None
9,9,47113199999,True,297845,None


In [48]:
!zip -r met_csvs.zip met_csvs

  adding: met_csvs/ (stored 0%)
  adding: met_csvs/2_met.csv (deflated 90%)
  adding: met_csvs/19_met.csv (deflated 91%)
  adding: met_csvs/7_met.csv (deflated 92%)
  adding: met_csvs/8_met.csv (deflated 92%)
  adding: met_csvs/3_met.csv (deflated 90%)
  adding: met_csvs/11_met.csv (deflated 90%)
  adding: met_csvs/4_met.csv (deflated 91%)
  adding: met_csvs/18_met.csv (deflated 92%)
  adding: met_csvs/14_met.csv (deflated 90%)
  adding: met_csvs/17_met.csv (deflated 90%)
  adding: met_csvs/10_met.csv (deflated 90%)
  adding: met_csvs/5_met.csv (deflated 91%)
  adding: met_csvs/20_met.csv (deflated 91%)
  adding: met_csvs/9_met.csv (deflated 92%)
  adding: met_csvs/.ipynb_checkpoints/ (stored 0%)
  adding: met_csvs/6_met.csv (deflated 90%)
  adding: met_csvs/0_met.csv (deflated 92%)
  adding: met_csvs/21_met.csv (deflated 91%)
  adding: met_csvs/13_met.csv (deflated 90%)
  adding: met_csvs/16_met.csv (deflated 90%)
  adding: met_csvs/12_met.csv (deflated 90%)
  adding: met_csvs/1_met.c

In [49]:
files.download("met_csvs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>